# Rolling Average

In [10]:
import pandas as pd

In [11]:
# 1. We need the original, unmerged dataframe. 

df = pd.read_csv("../data/raw_games.csv")
df = df.sort_values(by=['GAME_DATE', 'GAME_ID']).reset_index(drop=True)
df

,SEASON_ID,TEAM_ID,TEAM_ABBREVIATION,TEAM_NAME,GAME_ID,GAME_DATE,MATCHUP,WL,MIN,FGM,...,REB,AST,STL,BLK,TOV,PF,PTS,PLUS_MINUS,VIDEO_AVAILABLE,Season
0,22021,1610612749,MIL,Milwaukee Bucks,22100001,2021-10-19,MIL vs. BKN,W,240,48,...,54,25,8,9,8,19,127,23,1,2021-22
1,22021,1610612751,BKN,Brooklyn Nets,22100001,2021-10-19,BKN @ MIL,L,240,37,...,44,19,3,9,13,17,104,-23,1,2021-22
2,22021,1610612744,GSW,Golden State Warriors,22100002,2021-10-19,GSW @ LAL,W,240,41,...,50,30,9,2,17,18,121,7,1,2021-22
3,22021,1610612747,LAL,Los Angeles Lakers,22100002,2021-10-19,LAL vs. GSW,L,240,45,...,45,21,7,4,18,25,114,-7,1,2021-22
4,22021,1610612754,IND,Indiana Pacers,22100003,2021-10-20,IND @ CHA,L,240,42,...,51,29,2,10,17,24,122,-1,1,2021-22
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
12295,22025,1610612762,UTA,Utah Jazz,22501198,2026-04-12,UTA @ LAL,L,240,41,...,42,25,13,4,16,14,107,-24,1,2025-26
12296,22025,1610612746,LAC,LA Clippers,22501199,2026-04-12,LAC vs. GSW,W,240,42,...,44,27,3,4,10,24,115,5,1,2025-26
12297,22025,1610612744,GSW,Golden State Warriors,22501199,2026-04-12,GSW @ LAC,L,240,36,...,47,24,3,3,10,21,110,-5,1,2025-26
12298,22025,1610612758,SAC,Sacramento Kings,22501200,2026-04-12,SAC @ POR,L,240,40,...,47,24,7,8,17,18,110,-12,1,2025-26


In [12]:
# 2. Define the dictionary of stats and their specific EWMA spans
variable_spans = {
    'PTS': 10, 'AST': 10, 'TOV': 10, 'PF': 10,  
    'FG_PCT': 15, 'FG3_PCT': 15, 'FT_PCT': 15,  
    'OREB': 15, 'DREB': 15,                     
    'STL': 15, 'BLK': 15                        
}

# 3. Create a function that loops over the dictionary
def calculate_variable_rolling(group, spans_dict):
    # Start the list with the identifying columns explicitly
    rolled_dfs = []
    
    for col, span in spans_dict.items():
        rolled_col = group[[col]].ewm(span=span, min_periods=1).mean().shift(1)
        rolled_col.columns = [f"{col}_EMA_{span}"]
        rolled_dfs.append(rolled_col)
        
    # Concatenate everything internally
    return pd.concat(rolled_dfs, axis=1)

# Now you just run the groupby
team_rolling_stats = df.groupby('TEAM_ABBREVIATION', group_keys=False).apply(
    calculate_variable_rolling, spans_dict=variable_spans
)

team_rolling_stats.head()


,PTS_EMA_10,AST_EMA_10,TOV_EMA_10,PF_EMA_10,FG_PCT_EMA_15,FG3_PCT_EMA_15,FT_PCT_EMA_15,OREB_EMA_15,DREB_EMA_15,STL_EMA_15,BLK_EMA_15
0,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
1,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
2,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
3,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
4,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN


In [13]:

# 5. Combine the rolling stats with the necessary identifiers
team_rolling_stats = pd.concat([df[['GAME_ID', 'TEAM_ABBREVIATION']], team_rolling_stats], axis=1)

team_rolling_stats.head(35)


,GAME_ID,TEAM_ABBREVIATION,PTS_EMA_10,AST_EMA_10,TOV_EMA_10,PF_EMA_10,FG_PCT_EMA_15,FG3_PCT_EMA_15,FT_PCT_EMA_15,OREB_EMA_15,DREB_EMA_15,STL_EMA_15,BLK_EMA_15
0,22100001,MIL,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
1,22100001,BKN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
2,22100002,GSW,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
3,22100002,LAL,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
4,22100003,IND,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
5,22100003,CHA,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
6,22100004,CHI,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
7,22100004,DET,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
8,22100005,BOS,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
9,22100005,NYK,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN


In [14]:
# 1. Load your intermediate games dataframe
games_df = pd.read_csv("../data/merged_games_intermediate.csv")

# 2. Add prefixes to our rolling stats so they don't overwrite each other
home_rolling = team_rolling_stats.add_prefix('HOME_')
away_rolling = team_rolling_stats.add_prefix('AWAY_')

# 3. Merge HOME stats based on GAME_ID and TEAM_ABBREVIATION
games_df = pd.merge(games_df, home_rolling, 
                    left_on=['GAME_ID', 'HOME_TEAM_ABBREVIATION'], 
                    right_on=['HOME_GAME_ID', 'HOME_TEAM_ABBREVIATION'], 
                    how='inner')

# 4. Merge AWAY stats based on GAME_ID and TEAM_ABBREVIATION
games_df = pd.merge(games_df, away_rolling, 
                    left_on=['GAME_ID', 'AWAY_TEAM_ABBREVIATION'], 
                    right_on=['AWAY_GAME_ID', 'AWAY_TEAM_ABBREVIATION'], 
                    how='inner')

# 5. Clean up redundant ID columns created by the merge
games_df = games_df.drop(columns=['HOME_GAME_ID', 'AWAY_GAME_ID'])

print(f"Shape after merge: {games_df.shape}")
games_df.head()


Shape after merge: (6140, 69)


,HOME_TEAM_ABBREVIATION,GAME_ID,HOME_GAME_DATE,HOME_MATCHUP,HOME_WL,HOME_MIN,HOME_FGM,HOME_FGA,HOME_FG_PCT,HOME_FG3M,...,AWAY_AST_EMA_10,AWAY_TOV_EMA_10,AWAY_PF_EMA_10,AWAY_FG_PCT_EMA_15,AWAY_FG3_PCT_EMA_15,AWAY_FT_PCT_EMA_15,AWAY_OREB_EMA_15,AWAY_DREB_EMA_15,AWAY_STL_EMA_15,AWAY_BLK_EMA_15
0,LAL,22100002,2021-10-19,LAL vs. GSW,0,240,45,95,0.474,15,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
1,MIL,22100001,2021-10-19,MIL vs. BKN,1,240,48,105,0.457,17,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
2,CHA,22100003,2021-10-20,CHA vs. IND,1,240,46,107,0.430,13,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
3,DET,22100004,2021-10-20,DET vs. CHI,0,240,36,90,0.400,6,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
4,MEM,22100007,2021-10-20,MEM vs. CLE,1,240,53,100,0.530,14,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN


In [15]:
# The raw columns that cause 100% target leakage
leaky_cols = [
    'HOME_MIN', 'HOME_FGM', 'HOME_FGA', 'HOME_FG_PCT', 'HOME_FG3M', 'HOME_FG3A', 'HOME_FG3_PCT', 
    'HOME_FTM', 'HOME_FTA', 'HOME_FT_PCT', 'HOME_OREB', 'HOME_DREB', 'HOME_REB', 'HOME_AST', 
    'HOME_STL', 'HOME_BLK', 'HOME_TOV', 'HOME_PF', 'HOME_PTS',
    
    'AWAY_MIN', 'AWAY_FGM', 'AWAY_FGA', 'AWAY_FG_PCT', 'AWAY_FG3M', 'AWAY_FG3A', 'AWAY_FG3_PCT', 
    'AWAY_FTM', 'AWAY_FTA', 'AWAY_FT_PCT', 'AWAY_OREB', 'AWAY_DREB', 'AWAY_REB', 'AWAY_AST', 
    'AWAY_STL', 'AWAY_BLK', 'AWAY_TOV', 'AWAY_PF', 'AWAY_PTS'
]

# Drop the leaky columns correctly
games_df = games_df.drop(columns=leaky_cols)

# Drop the first ~10 games for each team that have NaN rolling averages
games_df = games_df.dropna().reset_index(drop=True)

# Look at your final clean matrix
print(games_df.shape)
games_df.head()


(6124, 31)


,HOME_TEAM_ABBREVIATION,GAME_ID,HOME_GAME_DATE,HOME_MATCHUP,HOME_WL,AWAY_TEAM_ABBREVIATION,AWAY_GAME_DATE,AWAY_MATCHUP,AWAY_WL,HOME_PTS_EMA_10,...,AWAY_AST_EMA_10,AWAY_TOV_EMA_10,AWAY_PF_EMA_10,AWAY_FG_PCT_EMA_15,AWAY_FG3_PCT_EMA_15,AWAY_FT_PCT_EMA_15,AWAY_OREB_EMA_15,AWAY_DREB_EMA_15,AWAY_STL_EMA_15,AWAY_BLK_EMA_15
0,PHI,22100021,2021-10-22,PHI vs. BKN,0,BKN,2021-10-22,BKN @ PHI,1,117.0,...,19.0,13.0,17.0,0.440,0.531,0.565,5.0,39.0,3.0,9.0
1,WAS,22100019,2021-10-22,WAS vs. IND,1,IND,2021-10-22,IND @ WAS,0,98.0,...,29.0,17.0,24.0,0.467,0.362,0.875,8.0,43.0,2.0,10.0
2,SAC,22100026,2021-10-22,SAC vs. UTA,0,UTA,2021-10-22,UTA @ SAC,1,124.0,...,18.0,10.0,19.0,0.440,0.298,0.867,12.0,41.0,6.0,5.0
3,ORL,22100018,2021-10-22,ORL vs. NYK,0,NYK,2021-10-22,NYK @ ORL,1,97.0,...,27.0,19.0,22.0,0.486,0.378,0.704,7.0,48.0,9.0,10.0
4,LAL,22100025,2021-10-22,LAL vs. PHX,0,PHX,2021-10-22,PHX @ LAL,1,114.0,...,23.0,18.0,18.0,0.414,0.378,0.706,11.0,34.0,9.0,3.0


In [16]:
games_df.to_csv("../data/final_training_data.csv", index=False)


In [17]:
# Extract only the identifying columns
schedule_df = games_df[['GAME_ID', 'HOME_GAME_DATE', 'HOME_TEAM_ABBREVIATION', 'AWAY_TEAM_ABBREVIATION']]

# Rename the date column so it looks cleaner
schedule_df = schedule_df.rename(columns={'HOME_GAME_DATE': 'GAME_DATE'})

# Save it as a separate lookup table
schedule_df.to_csv("../data/game_schedule.csv", index=False)

schedule_df.head()


,GAME_ID,GAME_DATE,HOME_TEAM_ABBREVIATION,AWAY_TEAM_ABBREVIATION
0,22100021,2021-10-22,PHI,BKN
1,22100019,2021-10-22,WAS,IND
2,22100026,2021-10-22,SAC,UTA
3,22100018,2021-10-22,ORL,NYK
4,22100025,2021-10-22,LAL,PHX
